Import big matrix

In [4]:
import pandas as pd

bm_og = pd.read_csv(
    '/home/sulay/deep-linear-bandits/kuairec/data/big_matrix.csv',
        usecols=[
            'user_id',
            'video_id',
            'watch_ratio'
        ]
    ).sort_values(by=['user_id', 'video_id'])

bm_og

,user_id,video_id,watch_ratio
312,0,42,1.098951
292,0,67,2.759635
275,0,80,1.188017
61,0,110,1.408627
318,0,128,1.281867
...,...,...,...
12530627,7175,10552,0.147431
12530710,7175,10572,0.293611
12530711,7175,10572,0.293611
12530717,7175,10589,0.560359


In [5]:
bm_og.duplicated(subset=['user_id', 'video_id']).sum()

np.int64(2229837)

In [6]:
bm_f = bm_og.copy()

bm_f = bm_f.groupby(by=['user_id', 'video_id'], as_index=False).max()

bm_f

,user_id,video_id,watch_ratio
0,0,42,1.098951
1,0,67,2.759635
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


In [7]:
bm_f.groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,80,80
4604,125,125
3935,151,151
3175,155,155
268,156,156
...,...,...
6565,3341,3341
816,3348,3348
5412,3353,3353


In [8]:
bm_f[bm_f.watch_ratio < 2.0].groupby(by='user_id').count().sort_values(by='watch_ratio')

,video_id,watch_ratio
user_id,,
3116,63,63
4604,102,102
3935,121,121
3175,133,133
4812,138,138
...,...,...
6565,3295,3295
2735,3303,3303
5412,3308,3308


In [9]:
bm_f.groupby(by='video_id').count().sort_values(by='watch_ratio')

,user_id,watch_ratio
video_id,,
11,1,1
4943,1,1
40,1,1
43,1,1
10705,1,1
...,...,...
1507,5168,5168
8145,5175,5175
1037,5211,5211


Filter for the strong positive interactions to train on

In [10]:
bm = bm_f.copy()

bm = bm[bm.watch_ratio >= 2.0]

bm

,user_id,video_id,watch_ratio
1,0,67,2.759635
6,0,133,2.458447
10,0,152,2.326087
11,0,154,4.353647
15,0,171,33.276021
...,...,...,...
10300855,7175,9811,2.232981
10300862,7175,9841,2.167814
10300910,7175,10130,15.828342
10300955,7175,10408,9.241597


In [11]:
liked_counts = bm.groupby(by='user_id')['user_id'].count()

liked_counts

user_id
0       258
1       166
2        38
3       163
4        51
       ... 
7171    134
7172    271
7173     95
7174     37
7175    223
Name: user_id, Length: 7175, dtype: int64

In [12]:
liked_counts.describe()

count    7175.000000
mean      117.967108
std       101.901022
min         1.000000
25%        43.000000
50%        88.000000
75%       162.000000
max      1002.000000
Name: user_id, dtype: float64

In [13]:
bm_f[bm_f.watch_ratio < 2.0]

,user_id,video_id,watch_ratio
0,0,42,1.098951
2,0,80,1.188017
3,0,110,1.408627
4,0,128,1.281867
5,0,130,0.079565
...,...,...,...
10300964,7175,10519,1.107321
10300965,7175,10552,0.147431
10300966,7175,10572,0.293611
10300967,7175,10589,0.560359


Split into training & validation

In [14]:
from sklearn.model_selection import train_test_split

low = bm.groupby('user_id')['user_id'].transform('count') < 5

nonlow = bm[~low]
bm_train, bm_val = train_test_split(
    nonlow,
    train_size=0.8,
    shuffle=True,
    stratify=nonlow['user_id']
)
bm_train = pd.concat([bm_train, bm[low]])

bm_train, bm_val

(          user_id  video_id  watch_ratio
 8609713      5974      6028     2.646447
 10210433     7107       463     3.895321
 8309842      5766      8732     2.099693
 7513533      5199      3065     7.166466
 3863618      2705      3179     2.430638
 ...           ...       ...          ...
 9176402      6384      3632     3.930262
 9555229      6658      3400     2.603915
 9555261      6658      4374     2.519602
 9555297      6658      5047     2.600979
 9555519      6658     10653     2.045629
 
 [677144 rows x 3 columns],
          user_id  video_id  watch_ratio
 7742795     5356      7504     2.911438
 7676081     5309      1434     2.023584
 3495806     2434      5711     2.633462
 4387563     3063      5536     6.212684
 569655       380      7049     2.008343
 ...          ...       ...          ...
 7296980     5054      1110     2.156699
 1574102     1080       517     2.496561
 3193176     2223      8023     2.201401
 2584669     1790      8814     3.280000
 3853917     26

Convert to PyTorch dataset format

In [15]:
from torch.utils.data import Dataset
import torch
import numpy as np

K = 5

class KRDataset(Dataset):
    def __init__(self, data):
        self.user_ids = data['user_id'].to_numpy()
        self.item_ids = data['video_id'].to_numpy()
        
        # Compute user positive sets for rejection sampling negatives
        self.positives = [set() for _ in range(7176)]
        for i in range(len(self.user_ids)):
            self.positives[self.user_ids[i]].add(self.item_ids[i])

    def __len__(self):
        return len(self.user_ids)
    
    def __getitem__(self, idx):
        # Rejection sample negatives
        negatives = []
        while len(negatives) < K:
            n = np.random.randint(0, 7176)
            if n not in self.positives[self.user_ids[idx]]: negatives.append(n)

        return {
            'user_id': self.user_ids[idx],
            'pos_item_id': self.item_ids[idx],
            'neg_item_ids': torch.tensor(negatives)
        }

bm_t = KRDataset(bm_train)
bm_v = bm_val.drop(columns=['watch_ratio'])

print(len(bm_t))
print(len(bm_v))

677144
169270


Set up two-tower

In [16]:
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()

        self.u = nn.Embedding(7176, 32)
        self.i = nn.Embedding(10728, 32)

    def forward(self, user_id, pos_item_id, neg_item_ids):
        u_emb = self.u(user_id) # BATCH_SIZE x 1 -> BATCH_SIZE x EMB_SIZE
        pos_emb = self.i(pos_item_id) # BATCH_SIZE x 1 -> BATCH_SIZE x EMB_SIZE
        neg_embs = self.i(neg_item_ids) # BATCH_SIZE x NEGSAMPLES_NUM x 1 -> BATCH_SIZE x NEGSAMPLES_NUM x EMB_SIZE

        dots = (u_emb * pos_emb).sum(1, keepdim=True) # dot product between user and item for each batch -> BATCH_SIZE x 1

        u_emb = u_emb.unsqueeze(1) # BATCH_SIZE x EMB_SIZE -> BATCH_SIZE x 1 x EMB_SIZE
        neg_embs = neg_embs.transpose(1,2) # BATCH_SIZE x NEGSAMPLES_NUM x EMB_SIZE -> BATCH_SIZE x EMB_SIZE x NEGSAMPLES_NUM
        bmm = torch.bmm(u_emb, neg_embs) # matmul for each batch to get dot prod of user with all negatives -> BATCH_SIZE x 1 x NEGSAMPLES_NUM
        bmm = bmm.squeeze(1) # -> BATCH_SIZE x NEGSAMPLES_NUM

        # now combine the pos & neg for each batch
        return torch.cat([dots, bmm], dim=1) # BATCH_SIZE x (1 + NEGSAMPLES_NUM)

model = TwoTower().to(device)

model

TwoTower(
  (u): Embedding(7176, 32)
  (i): Embedding(10728, 32)
)

Train the two-tower and see validation loss each epoch

In [17]:
bm_train

,user_id,video_id,watch_ratio
8609713,5974,6028,2.646447
10210433,7107,463,3.895321
8309842,5766,8732,2.099693
7513533,5199,3065,7.166466
3863618,2705,3179,2.430638
...,...,...,...
9176402,6384,3632,3.930262
9555229,6658,3400,2.603915
9555261,6658,4374,2.519602
9555297,6658,5047,2.600979


In [43]:
# no data loader needed for validation; explicitly use bm_v since validation is not batchwise like training is
# will need to precompute necessary information for recall@k though
# 1. what positives are held out per user?
held_out = bm_v.groupby('user_id')['video_id'].apply(set).to_dict() # note not all users are in validation set of course; note the groupby auto sorts

# 2. training positives are already known via bm_train
all_users = torch.arange(7176, device=device)
all_items = torch.arange(10728, device=device)
rows = torch.tensor(bm_train['user_id'].values, device=device)
cols = torch.tensor(bm_train['video_id'].values, device=device)

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

bm_train_ldr = DataLoader(bm_t, batch_size=512, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs=10
for epoch in range(epochs):
    model.train()
    total_loss_t = 0
    for batch in tqdm(bm_train_ldr):
        opt.zero_grad()

        logits = model(
            batch['user_id'].to(device),
            batch['pos_item_id'].to(device),
            batch['neg_item_ids'].to(device)
        )
        target = torch.zeros(logits.size(0), dtype=torch.long, device=device)

        loss = loss_fn(logits, target)

        loss.backward()
        opt.step()

        total_loss_t += loss.item()
    
    print(f"End of epoch {epoch} average batch training loss: {total_loss_t / len(bm_train_ldr)}")

    # Compute Recall@K on validation set
    model.eval()
    with torch.no_grad():
        # Compute user embeddings for all users (will filter out for just validation set users later)
        u_embs = model.u(all_users)

        # Compute item embeddings for all items
        i_embs = model.i(all_items)

        # Compute dotprod scores
        scores = u_embs @ i_embs.T

        # Mask out positive interactions from training set
        scores[rows, cols] = -1e9

        # Mask out non-validation users
        scores = scores[list(held_out.keys()), :]

        # Top-K predictions
        K = 50
        topk = torch.topk(scores, K, dim=1).indices

        # Check for positives
        ho = list(held_out.values())
        counts = [
            sum(v in ho[i] for v in row)
            for i, row in enumerate(topk)
        ]
        print(counts)
    


  8%|▊         | 101/1323 [00:07<01:31, 13.37it/s]